# 2. 모델 학습
## SageMaker에서 YOLOv8 안전장비 감지 모델 학습

이 노트북에서는 다음을 수행합니다:
1. SageMaker 학습 작업 설정
2. 하이퍼파라미터 튜닝
3. 모델 학습 실행
4. 학습 결과 분석

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter
import boto3

session = sagemaker.Session()
bucket = session.default_bucket()
role = sagemaker.get_execution_role()
region = session.boto_region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")

In [ ]:
# 데이터 경로
prefix = 'construction-safety'
train_s3 = f's3://{bucket}/{prefix}/data/train'
val_s3 = f's3://{bucket}/{prefix}/data/val'

print(f"Train: {train_s3}")
print(f"Val: {val_s3}")

## 2.1 학습 작업 설정

In [ ]:
# 하이퍼파라미터
hyperparameters = {
    'epochs': 100,
    'batch-size': 16,
    'img-size': 640,
    'learning-rate': 0.01,
    'model-size': 'n',  # nano - Raspberry Pi 최적화
    'patience': 50,
    'optimizer': 'SGD'
}

# PyTorch Estimator
estimator = PyTorch(
    entry_point='train_safety_model.py',
    source_dir='../training',
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',  # GPU 인스턴스
    framework_version='2.0.0',
    py_version='py310',
    hyperparameters=hyperparameters,
    output_path=f's3://{bucket}/{prefix}/output',
    base_job_name='safety-detection',
    environment={
        'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:512'
    }
)

print("Estimator 설정 완료")

In [ ]:
# 학습 실행
estimator.fit({
    'train': train_s3,
    'val': val_s3
}, wait=True)

## 2.2 하이퍼파라미터 튜닝 (선택적)

In [ ]:
# 하이퍼파라미터 범위 정의
hyperparameter_ranges = {
    'learning-rate': ContinuousParameter(0.001, 0.1),
    'batch-size': IntegerParameter(8, 32)
}

# 튜너 설정
tuner = HyperparameterTuner(
    estimator,
    objective_metric_name='mAP50',
    objective_type='Maximize',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,
    max_parallel_jobs=2
)

print("하이퍼파라미터 튜너 설정 완료")

## 2.3 학습 결과 분석

In [ ]:
# 학습된 모델 경로
model_data = estimator.model_data
print(f"모델 아티팩트: {model_data}")